# 05 ResNet Training — Colab Clean

This notebook trains ResNet18 using the shared splits and saves model weights, metrics, probabilities, and features for Week 3 fusion.

In [1]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA available: True
GPU: NVIDIA L4


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!rm -rf /content/movie_genre_classification
!git clone https://github.com/oluwoleadetifa/movie_genre_classification.git
%cd /content/movie_genre_classification

import sys
sys.path.append("/content/movie_genre_classification")

Cloning into 'movie_genre_classification'...
remote: Enumerating objects: 317, done.
remote: Counting objects: 100% (132/132), done.
remote: Compressing objects: 100% (118/118), done.
remote: Total 317 (delta 66), reused 37 (delta 14), pack-reused 185 (from 1)
Receiving objects: 100% (317/317), 1.46 MiB | 4.87 MiB/s, done.
Resolving deltas: 100% (156/156), done.
/content/movie_genre_classification


In [4]:
!pip install -q python-dotenv scikit-image kaggle datasets transformers scikit-learn pandas numpy pillow tqdm

In [5]:
from pathlib import Path
import shutil
import os

PROJECT_DIR = Path("/content/movie_genre_classification")
DRIVE_BASE = Path("/content/drive/MyDrive/Movie_Genre_Project")

# Adjust these only if your Drive folder names differ.
SPLITS_SRC = DRIVE_BASE / "splits"
PROCESSED_SRC = DRIVE_BASE / "dataset_processed" / "movies_with_posters.csv"
POSTERS_SRC = DRIVE_BASE / "posters"

SPLITS_DST = PROJECT_DIR / "data" / "splits"
PROCESSED_DST = PROJECT_DIR / "data" / "processed" / "movies_with_posters.csv"
POSTERS_DST = PROJECT_DIR / "data" / "processed" / "posters"

SPLITS_DST.mkdir(parents=True, exist_ok=True)
PROCESSED_DST.parent.mkdir(parents=True, exist_ok=True)

# Copy shared splits only. Do not regenerate random splits.
if SPLITS_SRC.exists():
    for name in ["train.csv", "val.csv", "test.csv"]:
        shutil.copy2(SPLITS_SRC / name, SPLITS_DST / name)
else:
    print("Drive splits folder not found. Using existing repo splits if present:", SPLITS_DST)

if PROCESSED_SRC.exists():
    shutil.copy2(PROCESSED_SRC, PROCESSED_DST)
else:
    print("Processed CSV not found in Drive. Continuing if your repo already has it.")

if POSTERS_SRC.exists():
    shutil.copytree(POSTERS_SRC, POSTERS_DST, dirs_exist_ok=True)
else:
    print("Poster folder not found at", POSTERS_SRC)
    print("Continuing if poster paths in CSV already point to valid files.")

print("Splits:")
!ls -lh /content/movie_genre_classification/data/splits

Poster folder not found at /content/drive/MyDrive/Movie_Genre_Project/posters
Continuing if poster paths in CSV already point to valid files.
Splits:
total 1.6M
-rw------- 1 root root 269K Apr 22 02:30 test.csv
-rw------- 1 root root 1.1M Apr 22 02:31 train.csv
-rw------- 1 root root 196K Apr 22 02:31 val.csv


In [8]:
from pathlib import Path

models_dir = Path("/content/movie_genre_classification/src/models")
models_dir.mkdir(parents=True, exist_ok=True)

file_path = models_dir / "resnet_image.py"

file_path.write_text("""
# paste your FULL resnet_image.py code here
""")

print("Wrote:", file_path)

Wrote: /content/movie_genre_classification/src/models/resnet_image.py


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path("/content/movie_genre_classification")
SPLITS_DIR = PROJECT_DIR / "data" / "splits"

OLD_PREFIX = "/Users/oluwoleadetifa/Library/CloudStorage/GoogleDrive-adetifaoluwole@gmail.com/My Drive/Movie_Genre_Project"
NEW_PREFIX = "/content/drive/MyDrive/Movie_Genre_Project"

for split in ["train", "val", "test"]:
    csv_path = SPLITS_DIR / f"{split}.csv"
    df = pd.read_csv(csv_path)

    path_col = "image_path"

    df[path_col] = df[path_col].astype(str).str.replace(
        OLD_PREFIX,
        NEW_PREFIX,
        regex=False
    )

    df.to_csv(csv_path, index=False)
    print(f"Updated {split}.csv")
    print(df[path_col].iloc[0])

Updated train.csv
/content/drive/MyDrive/Movie_Genre_Project/dataset_raw/IMDB four_genre_posters/Action/tt20222202.jpg
Updated val.csv
/content/drive/MyDrive/Movie_Genre_Project/dataset_raw/IMDB four_genre_posters/Action/tt17156822.jpg
Updated test.csv
/content/drive/MyDrive/Movie_Genre_Project/dataset_raw/IMDB four_genre_posters/Action/tt13462900.jpg


In [17]:
from pathlib import Path
import pandas as pd

df = pd.read_csv("/content/movie_genre_classification/data/splits/train.csv")
sample_path = Path(df["image_path"].iloc[0])

print(sample_path)
print("Exists?", sample_path.exists())

/content/drive/MyDrive/Movie_Genre_Project/dataset_raw/IMDB four_genre_posters/Action/tt20222202.jpg
Exists? True


In [18]:
# Train ResNet18 and save fusion-ready outputs.
!python -m src.models.resnet_image

Using device: cuda

Epoch 1/10
Train Loss: 1.1224 | Train Acc: 0.5396 | Train Macro F1: 0.4847
Val   Loss: 1.2185 | Val   Acc: 0.5631 | Val   Macro F1: 0.5322
Saved new best model.

Epoch 2/10
Train Loss: 0.4062 | Train Acc: 0.8750 | Train Macro F1: 0.8531
Val   Loss: 0.9954 | Val   Acc: 0.5825 | Val   Macro F1: 0.5369
Saved new best model.

Epoch 3/10
Train Loss: 0.1705 | Train Acc: 0.9792 | Train Macro F1: 0.9754
Val   Loss: 0.8268 | Val   Acc: 0.6893 | Val   Macro F1: 0.6067
Saved new best model.

Epoch 4/10
Train Loss: 0.0763 | Train Acc: 0.9917 | Train Macro F1: 0.9885
Val   Loss: 0.7886 | Val   Acc: 0.6796 | Val   Macro F1: 0.5929

Epoch 5/10
Train Loss: 0.0551 | Train Acc: 0.9938 | Train Macro F1: 0.9944
Val   Loss: 0.8830 | Val   Acc: 0.6408 | Val   Macro F1: 0.5840

Epoch 6/10
Train Loss: 0.0306 | Train Acc: 1.0000 | Train Macro F1: 1.0000
Val   Loss: 0.9496 | Val   Acc: 0.6311 | Val   Macro F1: 0.5663

Epoch 7/10
Train Loss: 0.0202 | Train Acc: 1.0000 | Train Macro F1: 1.0000

In [10]:
# Check saved outputs.
!find /content/movie_genre_classification/outputs -maxdepth 3 -type f | sort

/content/movie_genre_classification/outputs/figures/.gitkeep
/content/movie_genre_classification/outputs/logs/.gitkeep
/content/movie_genre_classification/outputs/metrics/.gitkeep
/content/movie_genre_classification/outputs/models/.gitkeep


In [11]:
# Optional: copy outputs back to Drive so they survive Colab runtime reset.
from pathlib import Path
import shutil

DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/Movie_Genre_Project/outputs")
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

shutil.copytree("/content/movie_genre_classification/outputs", DRIVE_OUTPUT_DIR, dirs_exist_ok=True)
print("Copied outputs to:", DRIVE_OUTPUT_DIR)

Copied outputs to: /content/drive/MyDrive/Movie_Genre_Project/outputs


In [19]:
!mkdir -p "/content/drive/MyDrive/Movie_Genre_Project/outputs/models"
!cp /content/movie_genre_classification/outputs/models/*.pth "/content/drive/MyDrive/Movie_Genre_Project/outputs/models/"